# C: Guidebook and test sentences (Amen)

This notebook builds and checks the two files Person C owns:

| File | What it is | Used by |
|---|---|---|
| `data/guidebook.md` | 25 neutral-wording rules (`## R01 ...`), retrieved one entry at a time by RAG | Yasa (`src/rag.py`), Madina (rewriter prompt) |
| `data/test_sentences.csv` | 50 real biased FigNews sentences (25 per side) with a neutral version and the rules they need | Yasa (recall test), Madina (rewriter test) |

The work happened in two rounds:

- **Round 1 (tuning set, ids 1 to 19):** find sentences, write the guidebook, and reword rules until retrieval works.
- **Round 2 (held-out set, ids 20 to 50):** add 31 new sentences **without changing any rule**, to measure how well
  the guidebook works on sentences it was not tuned on.

**Sections**

| Part | Section | What it does |
|---|---|---|
| Setup | C0 | Team setup and paths |
| Data | C1 | FigNews structure and label names |
| | C2 | Posts where annotators agree (English only) |
| Round 1 | C3 | Candidate sentences for both sides |
| | C4 | Test sentences 1 to 19 |
| | C5 | Guidebook format check and where the rules come from |
| | C6 | Retrieval test on the tuning set |
| Round 2 | C7 | Held-out sentences 20 to 50 and their retrieval test |
| Wrap-up | C8 | Iteration log and findings |
| | C9 | Symmetry review sheet |
| | C10 | Save the final files |

Runs on CPU; no GPU needed. Constants come from `config`; outputs go to `ARTIFACTS` (Drive), never to the repo.

## C0. Setup
Run the two cells below in order. The first is a temporary workaround: Colab's Python 3.13 removed the `imp`
module that IPython's `autoreload` still imports. The second is the unchanged team setup cell.

In [ ]:
# Workaround for Colab (Python 3.13): IPython's autoreload still imports the removed "imp" module.
import sys, types, importlib
if "imp" not in sys.modules:
    _imp = types.ModuleType("imp")
    _imp.reload = importlib.reload
    sys.modules["imp"] = _imp

In [ ]:
# === TEAM SETUP CELL: run first. Same on Colab, Kaggle and local Jupyter. ===
# Do not edit this cell in your notebook; change src/ or requirements.txt in the repo instead.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/iibrohim-hash/debias.git"
BRANCH = "main"  # use your feature branch while developing, e.g. "iroda/detector"

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/debias")
elif IN_KAGGLE:
    REPO_DIR = Path("/kaggle/working/debias")
else:  # local Jupyter: notebook is inside repo/notebooks
    REPO_DIR = Path.cwd().parent if (Path.cwd().parent / "src" / "config.py").exists() else Path.cwd()

if IN_COLAB or IN_KAGGLE:
    if not (REPO_DIR / "src").exists():
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "-q"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-q", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

%load_ext autoreload
%autoreload 2

from src import config, utils, contract
utils.set_seed(config.SEED)
if utils.get_hf_token():  # from Colab/Kaggle secrets; never type a token here (public repo)
    os.environ["HF_TOKEN"] = utils.get_hf_token()
ARTIFACTS = utils.artifacts_dir()
print("repo:", REPO_DIR, "| artifacts:", ARTIFACTS)
utils.print_env()

In [ ]:
import re
import pandas as pd
from pathlib import Path
from src import rag

pd.set_option("display.max_colwidth", None)

# Where my draft guidebook lives while I work on it (outside the repo).
# Set to None to use the team file data/guidebook.md instead.
DRAFT_GUIDEBOOK = ARTIFACTS / "draft_guidebook.md"
if DRAFT_GUIDEBOOK and Path(DRAFT_GUIDEBOOK).exists():
    config.GUIDEBOOK_PATH = Path(DRAFT_GUIDEBOOK)   # src/rag.py reads this at call time
print("guidebook used:", config.GUIDEBOOK_PATH)

*Optional:* upload a new version of the draft guidebook from your computer. It replaces the file on Drive.
Set `UPLOAD = True` only when you have a new version.

In [ ]:
UPLOAD = False
if UPLOAD:
    import shutil
    from google.colab import files
    up = files.upload()                                 # pick draft_guidebook.md
    shutil.move(list(up.keys())[0], ARTIFACTS / "draft_guidebook.md")
    config.GUIDEBOOK_PATH = ARTIFACTS / "draft_guidebook.md"
    rag.rebuild_index()
    print("saved to", config.GUIDEBOOK_PATH)

## C1. FigNews: structure and label names
Loaded with the pinned revision from `config`, so everyone sees the same rows.
Each row is **one annotator's label for one post**, so a post appears once per annotator.

In [ ]:
from datasets import load_dataset

ds = load_dataset(config.FIGNEWS_DATASET, revision=config.FIGNEWS_REVISION)
df = ds["train"].to_pandas()
print(ds, "\n")
print("columns:", df.columns.tolist(), "\n")
print(df["subtask"].value_counts(), "\n")

bias = df[df["subtask"] == "Bias"]
print(bias["label"].value_counts())

**What this shows**

- We only use the `Bias` subtask.
- The two sides are the labels `Biased against Palestine` and `Biased against Israel`; `Unbiased` is the third class.
  `Biased against others`, `Biased against both ...`, `Unclear` and `Not Applicable` are not used.
- Every language has the same number of rows (each post was translated); we keep **English originals only**.

In [ ]:
# 5 random English posts per label, to see what each label looks like in practice.
en = bias[bias["source_language"] == "English"].drop_duplicates(subset="text")
for lab in en["label"].unique():
    rows = en[en["label"] == lab]
    print(f"\n===== {lab} ({len(rows)} unique posts) =====")
    for t in rows["text"].sample(min(5, len(rows)), random_state=config.SEED):
        print("-", t[:300])

**Observations**

- Single-annotator labels are noisy: some "biased" posts look like plain reporting. So we keep only posts
  where at least `config.MIN_ANNOTATOR_AGREEMENT` annotators agree (C2), the same rule the detector uses.
- Much FigNews bias is about **what** is reported (topic choice), not **how**. Our rewriter only changes
  wording, so test sentences must contain specific biased words.
- Direct quotes are common. The rewriter must never edit them (guidebook rule R13).

## C2. Posts where annotators agree (English only)

In [ ]:
en_all = bias[bias["source_language"] == "English"]
by_post = en_all.groupby("text")["label"]
agreed = pd.DataFrame({
    "label": by_post.agg(lambda s: s.value_counts().idxmax()),   # majority label
    "votes": by_post.agg(lambda s: s.value_counts().max()),      # how many annotators chose it
}).query("votes >= @config.MIN_ANNOTATOR_AGREEMENT")

print(agreed["label"].value_counts())

The "against Israel" class is much smaller than "against Palestine", and more of its posts are biased by topic
rather than wording. C3 therefore uses a separate cue list for each side, so the test set can stay balanced.

# Round 1: tuning set (ids 1 to 19)

## C3. Finding candidate sentences for both sides
The cue words are only a **search aid** to find posts with word-level bias faster; every pick is judged by hand.

In [ ]:
CUES_AGAINST_PALESTINE = (r"terror|butcher|slaughter|massacre|barbar|savage|brutal|brazen|regime|animal|monster|"
                          r"so-called|evil|genocide|atrocit|horrific|clash|eliminat|wipe out|necessity|died|killed")
CUES_AGAINST_ISRAEL = (r"claim|alleg|supposed|so-called|refut|admit|disturbing|war on|relentless|indiscriminate|"
                       r"occupation|apartheid|martyr|massacre|ethnic cleansing|carnage|slaughter|onslaught|regime|"
                       r"colonial|deliberately")


def side_hits(label: str, cues: str) -> list[str]:
    """Agreed posts with this label that contain at least one cue word (stable order)."""
    posts = agreed[agreed["label"] == label].index
    return [t for t in posts if re.search(cues, t, re.I)]


hits_p = side_hits("Biased against Palestine", CUES_AGAINST_PALESTINE)
hits_i = side_hits("Biased against Israel", CUES_AGAINST_ISRAEL)
print(len(hits_p), "candidates against Palestine |", len(hits_i), "candidates against Israel")

SHOW = 25
for name, hits in (("against Palestine", hits_p), ("against Israel", hits_i)):
    print(f"\n===== {name} (first {SHOW}) =====")
    for i, t in enumerate(hits[:SHOW]):
        print(f"[{i}] {t[:300]}\n")

**Selection criteria:** the post has specific biased words (not just a biased topic); the biased words are in the
narrator's voice, not inside a quote; picks are balanced between the two sides.
The full posts of the picks are saved for provenance.

In [ ]:
PICK_P = [0, 5, 11, 14, 16, 19, 20, 22, 23]       # indices in hits_p
PICK_I = [0, 1, 6, 7, 8, 14, 15, 17, 20, 27]      # indices in hits_i

picked = pd.DataFrame(
    [{"side": "against_palestine", "text": hits_p[i]} for i in PICK_P]
    + [{"side": "against_israel", "text": hits_i[i]} for i in PICK_I]
)
picked.to_csv(ARTIFACTS / "candidate_sentences.csv", index=False)
print(picked["side"].value_counts())

## C4. Test sentences 1 to 19
Each picked post is trimmed to the **one sentence** with the biased wording (links and hashtags removed, real
names kept, since the rewriter sees real text). For each: a neutral version that changes wording only, the bias
type, and the guidebook rules it needs (`expected_rules`, separated by `;`).

Decisions made while writing the neutral versions:

- **Keep the actor.** "the Israeli war on Gaza has killed ..." becomes "Israel's military campaign in Gaza has killed ...",
  not "the Israel-Hamas war has killed ...", which would hide who did the killing (R08, R21).
- **Keep consensus terms.** In 17, "stolen" is replaced but "occupied" and "illegal according to international law"
  stay (R14, no false balance).
- **Same rule for both sides.** "massacre" is replaced in 7 and 18; "claims" becomes "says" or "statements"
  in 11, 12, 13, 14 and 19.
- **Facts are not touched.** In 10, "the ICJ has found Israel accountable" overstates the ruling, but that is a
  factual problem, not wording, so the rewriter must leave it (a known limitation).

In [ ]:
TEST_ROWS = [
    # id, side, biased, neutral, bias type, expected rules
    (1, "against_palestine",
     "\"Friends\" actor David Schwimmer called out some advocates against sexual violence, arguing that they ignored reports of Hamas terrorists' sex crimes on Oct. 7.",
     "\"Friends\" actor David Schwimmer called out some advocates against sexual violence, arguing that they ignored reports of sex crimes by Hamas members on Oct. 7.",
     "loaded label", "R02"),
    (2, "against_palestine",
     "Netta Atzili, whose parents were kidnapped by Hamas, delivers an emotional response to those doubting the atrocities committed by terrorists.",
     "Netta Atzili, whose parents were kidnapped by Hamas, delivers an emotional response to those doubting the atrocities committed by Hamas members.",
     "loaded label", "R02"),
    (3, "against_palestine",
     "While recovering from Israel's retaliatory airstrikes, Gaza residents expressed surprise and shock at the brazen attack by Hamas on Israeli soil.",
     "While recovering from Israel's airstrikes, Gaza residents expressed surprise and shock at the attack by Hamas on Israeli soil.",
     "causation framing; intensifier", "R11;R03"),
    (4, "against_palestine",
     "A Columbia University Professor slammed \"coward\" university professors over their responses -- or lack thereof -- to \"pro-terror student organizations\" in the wake of the mass murder of Jews in Israel by Hamas.",
     "A Columbia University Professor slammed \"coward\" university professors over their responses -- or lack thereof -- to \"pro-terror student organizations\" in the wake of the killing of Jews in Israel by Hamas.",
     "emotive term", "R18"),
    (5, "against_palestine",
     "A Miami dentist seen ripping down posters of Israelis who went missing during the brutal Hamas attack on October 7 has been fired.",
     "A Miami dentist seen ripping down posters of Israelis who went missing during the Hamas attack on October 7 has been fired.",
     "intensifier", "R03"),
    (6, "against_palestine",
     "A majority of Americans support Israel in its fight against Hamas following a horrific terrorist attack.",
     "A majority of Americans support Israel in its fight against Hamas following an attack.",
     "intensifier; loaded label", "R03;R02"),
    (7, "against_palestine",
     "A new restaurant in Jordan named \"October 7\" in apparent celebration of Hamas' massacre of innocent civilians in Israel sparked outrage this week.",
     "A new restaurant in Jordan named \"October 7\" in apparent celebration of Hamas' attack on civilians in Israel sparked outrage this week.",
     "massacre; intensifier", "R18;R03"),
    (8, "against_palestine",
     "A synagogue in southern Israel is in ruins after a direct hit from a rocket reportedly fired from a terrorist in Gaza.",
     "A synagogue in southern Israel is in ruins after a direct hit from a rocket reportedly fired from Gaza.",
     "evidence-only; loaded label", "R01;R02"),
    (9, "against_palestine",
     "A three-year-old American orphan is amongst the hostages being held by Hamas terrorists after their barbaric October 7 raid on Israel.",
     "A three-year-old American orphan is amongst the hostages being held by Hamas after its October 7 raid on Israel.",
     "loaded label; dehumanizing intensifier", "R02;R04"),
    (10, "against_israel",
     "After three months of relentless bombing, and after over 26,000 Palestinians have lost their lives, the ICJ has found Israel accountable.",
     "After three months of bombing, and after over 26,000 Palestinians have lost their lives, the ICJ has found Israel accountable.",
     "intensifier", "R03"),
    (11, "against_israel",
     "An Israeli journalist from Channel 13 News, refuted claims made by the military regarding alleged crimes committed by Hamas, which were published around the world.",
     "An Israeli journalist from Channel 13 News, disputed statements made by the military regarding alleged crimes committed by Hamas, which were published around the world.",
     "presupposing verb; claims vs says", "R06;R05"),
    (12, "against_israel",
     "British-born Israeli soldier Levi Simon recorded a video of him searching Gaza homes for supposed explosives and guns, claiming to find money instead.",
     "British-born Israeli soldier Levi Simon recorded a video of him searching Gaza homes for explosives and guns, saying he found money instead.",
     "skeptic word; claims vs says", "R07;R05"),
    (13, "against_israel",
     "David Hearst, editor-in-chief of Middle East Eye, provides a critical assessment of the 100-day mark of Israel's war on Gaza, exposing a stark contrast between official US and Israeli claims and the grim reality on the ground.",
     "David Hearst, editor-in-chief of Middle East Eye, provides a critical assessment of the 100-day mark of Israel's military campaign in Gaza, describing what he sees as a contrast between official US and Israeli statements and conditions on the ground.",
     "presupposing verb; intensifier; war framing", "R06;R03;R21"),
    (14, "against_israel",
     "Israel has launched a military operation at the hospital, which it claims is a Hamas base.",
     "Israel has launched a military operation at the hospital, which it says is a Hamas base.",
     "claims vs says", "R05"),
    (15, "against_israel",
     "In two months, the Israeli war on Gaza has killed over 15,500 civilians, including over 6,600 children.",
     "In two months, Israel's military campaign in Gaza has killed over 15,500 civilians, including over 6,600 children.",
     "war framing", "R21"),
    (16, "against_israel",
     "Motaz Azaiza, a Gaza photographer who amassed a large following on social media for his coverage of Israel's war on the besieged strip, is to stop posting because of the risks.",
     "Motaz Azaiza, a Gaza photographer who amassed a large following on social media for his coverage of Israel's military campaign in the besieged strip, is to stop posting because of the risks.",
     "war framing", "R21"),
    (17, "against_israel",
     "One of the reasons Hamas gave for its surprise attack on Israel was the ongoing military occupation of Palestine's West Bank, where over 700,000 Israeli settlers live on stolen Palestinian land.",
     "One of the reasons Hamas gave for its surprise attack on Israel was the ongoing military occupation of Palestine's West Bank, where over 700,000 Israeli settlers live on occupied Palestinian land.",
     "loaded word; false balance", "R14"),
    (18, "against_israel",
     "Roaa Hammam Ismail Haniyeh, a medical student at the Islamic University of Gaza and the granddaughter of Hamas leader Ismail Haniyeh, was killed in the Shaheen family massacre in the Gaza Strip.",
     "Roaa Hammam Ismail Haniyeh, a medical student at the Islamic University of Gaza and the granddaughter of Hamas leader Ismail Haniyeh, was killed along with members of the Shaheen family in the Gaza Strip.",
     "massacre", "R18"),
    (19, "against_israel",
     "Nick Maynard, an NHS surgeon from Britain who worked in Gaza's hospitals for over a decade, discredits Israel's claims at the International Court of Justice that they safeguard hospitals.",
     "Nick Maynard, an NHS surgeon from Britain who worked in Gaza's hospitals for over a decade, disputes Israel's statements at the International Court of Justice that they safeguard hospitals.",
     "presupposing verb; claims vs says", "R06;R05"),
]

tests = pd.DataFrame(TEST_ROWS, columns=["id", "side", "biased", "neutral", "bias_type", "expected_rules"])
tests.insert(1, "conflict", "israel_palestine")
assert tests["id"].is_unique and set(tests["conflict"]) <= set(config.CONFLICTS)
print(tests["side"].value_counts())
tests[["id", "side", "bias_type", "expected_rules"]]

## C5. Guidebook: load and format check
`src/rag.py` splits the guidebook at `## ` headings and takes the second word as the rule id, so every entry must
look like `## R07 Title` and contain `Rule:`, `Biased:` and `Neutral:` lines.

In [ ]:
entries = rag.load_entries(config.GUIDEBOOK_PATH)
ids = [rag.entry_id(e) for e in entries]

problems = []
for e in entries:
    lines = e.splitlines()
    if not re.match(r"## R\d\d \S", lines[0]):
        problems.append(f"bad heading: {lines[0]}")
    for field in ("Rule:", "Biased:", "Neutral:"):
        if sum(l.startswith(field) for l in lines) != 1:
            problems.append(f"{lines[0]}: needs exactly one '{field}' line")
if ids[0] != config.ALWAYS_INCLUDED_RULE:
    problems.append("R01 (evidence-only) must be the first entry")
if len(set(ids)) != len(ids):
    problems.append("duplicate rule ids")
unknown = {r for rs in tests["expected_rules"] for r in rs.split(";")} - set(ids)
if unknown:
    problems.append(f"test sentences use rules that do not exist: {sorted(unknown)}")

print(len(entries), "entries:", " ".join(ids))
print("format OK" if not problems else "\n".join(problems))

In [ ]:
# The rules at a glance: id and title.
pd.DataFrame({"id": ids, "title": [e.splitlines()[0].split(maxsplit=2)[2] for e in entries]})

### Where the rules come from
FigNews was used to **find** which bias types occur and to **test** retrieval. The rules themselves rest on
outside standards, so they are not tuned to one dataset and do not depend only on our own views:

- **Editorial standards** decide wording: Reuters Handbook of Journalism, BBC Editorial Guidelines,
  Wikipedia's "Words to watch" guideline.
- **International law** decides who can apply legal labels: Genocide Convention, Rome Statute, Hague Regulations
  (occupation), ICCPR Art. 20 and the UN Rabat Plan of Action (incitement and dehumanization).

The full rule-to-source table is at the top of the guidebook, above the first entry, so RAG never retrieves it.
Rules marked "Team rule" have no checked external source yet.

**Decision where sources disagree:** some governments legally designate groups as terrorist organizations, while
editorial standards such as the BBC's avoid "terrorist" in the narrator's voice. We follow the editorial standard
and allow a designation only as an attributed fact (R02).

**What this does and does not show:** the sources make the rules *justified* for any conflict. Whether they
*work* on new data is shown by the Russia-Ukraine sentences, which the guidebook is not tuned on.

In [ ]:
# Rule-to-source table from the guidebook preamble, and how many rules have an external source.
text = Path(config.GUIDEBOOK_PATH).read_text(encoding="utf-8")
src_rows = [l.strip("|").split("|", 1) for l in text.splitlines() if re.match(r"\| R\d\d \|", l)]
sources = pd.DataFrame([(r.strip(), s.strip()) for r, s in src_rows], columns=["rule", "source"])
assert set(sources["rule"]) == set(ids), "every rule needs a row in the sources table"
external = ~sources["source"].str.startswith("Team rule")
print(f"{external.sum()}/{len(sources)} rules have an external source")
sources

## C6. Retrieval test on the tuning set (recall@k)
A test sentence is a **hit** if at least one of its expected rules (R01 ignored, because it is always included)
is among the top k retrieved entries. Two kinds of query:

- `flagged`: only the biased words, which is what `src/pipeline.py` sends to `retrieve()`.
  Until Iroda's detector exists, the flagged words below are my own guess of what it will highlight.
- `sentence`: the whole biased sentence.

In [ ]:
FLAGGED = {
    1: "terrorists", 2: "terrorists", 3: "retaliatory brazen", 4: "mass murder", 5: "brutal",
    6: "horrific terrorist", 7: "massacre innocent", 8: "terrorist", 9: "terrorists barbaric",
    10: "relentless", 11: "refuted claims", 12: "supposed claiming", 13: "exposing stark claims grim reality",
    14: "claims", 15: "war on Gaza", 16: "war on", 17: "stolen", 18: "massacre", 19: "discredits claims",
}
assert set(FLAGGED) == set(tests["id"]), "run this before C7 adds ids 20+"


def recall(df: pd.DataFrame, mode: str, k: int) -> tuple[float, list]:
    hits, misses = 0, []
    for r in df.itertuples():
        expected = set(r.expected_rules.split(";")) - {config.ALWAYS_INCLUDED_RULE}
        query = FLAGGED[r.id] if mode == "flagged" else r.biased
        got = rag.retrieve_ids(query, k)
        if expected & set(got):
            hits += 1
        else:
            misses.append({"id": r.id, "expected": sorted(expected), "retrieved": got})
    return hits / len(df), misses


rag.rebuild_index()
results, all_misses = {}, {}
for mode in ("flagged", "sentence"):
    for k in (3, 4):
        score, misses = recall(tests, mode, k)
        results[f"{mode}_k{k}"] = round(score, 3)
        all_misses[f"{mode}_k{k}"] = misses
        print(f"{mode:8s} k={k}: recall = {round(score * len(tests))}/{len(tests)}")

pd.DataFrame([m | {"setting": s} for s, ms in all_misses.items() for m in ms if s.startswith("flagged")])

**What we learned in round 1**

- Whole-sentence queries mostly match the **topic** (Israel, Gaza, casualties), not the biased word, so they stay
  around 8/19. Flagged-word queries, which the pipeline uses, reach 18/19 to 19/19. The pipeline should keep
  querying with flagged words (shared with Yasa).
- A rule is found mainly through its `Rule:` line: a key word that appears only in the example is often missed.
- The remaining miss (6, "horrific terrorist") mixes two bias types; R04 and R20 are close neighbours, so it is left as is.
- **Caveat:** these scores are measured on the same 19 sentences used to tune the wording, so they are optimistic.
  Round 2 measures the guidebook on sentences it was not tuned on.

In [ ]:
out = utils.save_result(
    "guidebook_recall_tuning",
    results,
    extra={"n": len(tests), "n_rules": len(ids), "guidebook": str(config.GUIDEBOOK_PATH),
           "embedding_model": config.EMBEDDING_MODEL, "flagged_words": "hand-written stand-in for the detector",
           "note": "measured on the sentences used to tune the guidebook"},
)
print("saved:", out)

# Round 2: held-out set (ids 20 to 50)

## C7. Held-out sentences
31 new sentences, chosen **after** the guidebook was finished. No rule was changed after they were added, so their
recall is a fair estimate of how the guidebook does on new text.

Same selection criteria as round 1, plus: candidates come from `fignews.prepare()` (the same agreement rule and
cleaning as the detector), and posts from the detector's **test split** come first, because the detector was not
trained on them.

In [ ]:
# C7a. Round 2 candidates (target: 50 sentences in total)
from src import fignews

raw = fignews.load_raw()                   # bias subtask, pinned revision
posts, report = fignews.prepare(raw)       # same agreement, cleaning and split as the detector
print(report)

# Original English posts only, biased against one side
cand = posts[posts["is_original_english"] & posts["label"].isin(["against_SIDE_A", "against_SIDE_B"])].copy()

# Leave out the 19 we already have (match on the first 8 words of each biased sentence)
def _key(s, n=8):
    return " ".join(re.sub(r"[^\w]+", " ", s.lower()).split()[:n])
old_keys = [_key(s) for s in tests["biased"]]
norm = cand["text_en"].map(lambda t: " ".join(re.sub(r"[^\w]+", " ", t.lower()).split()))
cand = cand[~norm.map(lambda t: any(k in t for k in old_keys))]

# Wider cue lists than C3, so the smaller "against Israel" side gets enough candidates
CUES_P2 = (r"terror|butcher|slaughter|massacre|barbar|savage|brutal|brazen|regime|animal|monster|so-called|"
           r"evil|atrocit|horrific|jihad|human shield|hide behind|wipe out|eliminat|necessity|retaliat|"
           r"unprovoked|clash|claim|alleg|propaganda|depraved|monstrous|heinous")
CUES_I2 = (r"claim|alleg|supposed|so-called|refut|debunk|expos|admit|denied|disturbing|war on|relentless|"
           r"indiscriminate|onslaught|aggression|occupation forces|occupying|siege|besieged|collective punishment|"
           r"apartheid|colonial|regime|martyr|massacre|genocid|ethnic cleansing|carnage|slaughter|flatten|"
           r"carpet|deliberately|targeting civilians|brutal|horrific|horror|stolen|zionist")

SIDES = {"against_SIDE_B": ("against Palestine", CUES_P2), "against_SIDE_A": ("against Israel", CUES_I2)}
SHOW = 40
new_hits = {}
for label, (name, cues) in SIDES.items():
    rows = cand[cand["label"] == label]
    hits = rows[rows["text_en"].str.contains(cues, case=False, regex=True)]
    # Prefer the detector's test split: those posts were not used to train it
    hits = hits.sort_values(["split", "agreement"], key=lambda s: s.map({"test": 0, "val": 1, "train": 2}) if s.name == "split" else -s)
    new_hits[label] = hits.reset_index(drop=True)
    print(f"\n===== {name}: {len(rows)} new agreed posts, {len(hits)} with cue words (showing {min(SHOW, len(hits))}) =====")
    for i, r in new_hits[label].head(SHOW).iterrows():
        print(f"[{i}] ({r['split']}) {r['text_en'][:350]}\n")

Only 37 "against Israel" posts contain cue words, and most of those are quotes or plain reporting. C7b looks
further: (1) English posts without cue words, read by hand, and (2) FigNews's English translations, as a backup.
The 11 extra "against Israel" sentences (ids 40 to 50) all came from (1), so **no translations are used**.

In [ ]:
# C7b. More "against Israel" candidates: (1) English posts the cue words missed, (2) FigNews English translations
used_keys = [_key(s) for s in tests["biased"]]   # all current test sentences, incl. ids 20+
def _unused(df, col):
    n = df[col].map(lambda t: " ".join(re.sub(r"[^\w]+", " ", t.lower()).split()))
    return df[~n.map(lambda t: any(k in t for k in used_keys))]

ai = posts[posts["label"] == "against_SIDE_A"]

eng = _unused(ai[ai["is_original_english"]], "text_en")
eng = eng[~eng["text_en"].str.contains(CUES_I2, case=False, regex=True)]
print(f"===== (1) original English, no cue word: {len(eng)} posts =====")
for i, r in enumerate(eng.itertuples()):
    print(f"[E{i}] ({r.split}) {r.text_en[:300]}\n")

tr = _unused(ai[~ai["is_original_english"]], "text_en")
tr = tr[tr["text_en"].str.contains(CUES_I2, case=False, regex=True)]
print(f"\n===== (2) translated posts with cue words: {len(tr)} (showing 40) =====")
for i, r in enumerate(tr.head(40).itertuples()):
    print(f"[T{i}] ({r.source_language}, {r.split}) {r.text_en[:300]}\n")

**Choices made while writing the neutral versions**

- Quotes stay untouched (R13), e.g. "went through hell" in 30 and Hamas's 'aggression' in 38.
- No actor is added: in 23 and 31 the text never says which group is responsible, so the neutral versions do not name one (R01).
- Claims are attributed to the source the text names, e.g. "he says" in 29 and "it says" in 44 (R10).
- The same rule now appears on both sides: "claims" in 27 vs 14 and 19; "war on" in 26 vs 15, 16, 36 and 37.

In [ ]:
# C7c. Held-out test sentences 20 to 50 (added after the guidebook was finished)
NEW_ROWS = [
    # id, side, biased, neutral, bias type, expected rules
    (20, "against_palestine",
     "As a red alert alarm sounded in the busy hotel dining hall, many of the children scattered in fear, as the horrors of the Hamas massacre on October 7 came flooding back.",
     "As a red alert alarm sounded in the busy hotel dining hall, many of the children scattered in fear, as memories of the Hamas attack on October 7 came flooding back.",
     "intensifier; massacre", "R18;R03"),
    (21, "against_palestine",
     "Former Alaska Gov. Sarah Palin lamented Monday on NEWSMAX about all of the \"fake feminists\" marching for Hamas amid the terrorist group's war with Israel in the Gaza Strip.",
     "Former Alaska Gov. Sarah Palin lamented Monday on NEWSMAX about all of the \"fake feminists\" marching for Hamas amid the group's war with Israel in the Gaza Strip.",
     "loaded label", "R02"),
    (22, "against_palestine",
     "BBC's Jeremy Bowen reports from Israeli border village where Hamas massacred families in their homes.",
     "BBC's Jeremy Bowen reports from Israeli border village where Hamas killed families in their homes.",
     "massacre", "R18"),
    (23, "against_palestine",
     "HEARTBREAKING: Kfir, who was kidnapped along with his mother, father, and brother, is spending the day under the grip of terrorists in Gaza.",
     "Kfir, who was kidnapped along with his mother, father, and brother, is spending the day held captive in Gaza.",
     "loaded label; editorial commentary; evidence-only", "R02;R17;R01"),
    (24, "against_palestine",
     "Israel is expected to launch a ground invasion into Gaza after Hamas terror attacks.",
     "Israel is expected to launch a ground invasion into Gaza after Hamas attacks.",
     "contested label", "R20;R02"),
    (25, "against_palestine",
     "Yoav Atzmoni who is a member of the LGBTQ+ community wanted to send a message of hope to the people in Gaza living under Hamas brutality.",
     "Yoav Atzmoni who is a member of the LGBTQ+ community wanted to send a message of hope to the people in Gaza living under Hamas rule.",
     "intensifier", "R03"),
    (26, "against_palestine",
     "'The Morning Show' star blasts Hollywood's 'deafening silence' on antisemitism as Hamas terrorists wage war on Israel.",
     "'The Morning Show' star blasts Hollywood's 'deafening silence' on antisemitism during the war between Hamas and Israel.",
     "loaded label; war framing", "R02;R21"),
    (27, "against_palestine",
     "The Hamas terrorist organization released a report into its October 7 attack on Israel, claiming its fighters were committed to \"Islamic values\" and if civilians were targeted \"it happened accidentally.\"",
     "Hamas released a report into its October 7 attack on Israel, saying its fighters were committed to \"Islamic values\" and if civilians were targeted \"it happened accidentally.\"",
     "loaded label; claims vs says", "R02;R05"),
    (28, "against_palestine",
     "The war against Palestinian Hamas terrorists has united Jewish and Arab Israelis, after years of worries about tensions between the two.",
     "The war against the Palestinian group Hamas has united Jewish and Arab Israelis, after years of worries about tensions between the two.",
     "loaded label", "R02"),
    (29, "against_palestine",
     "Watch IDF Spokesperson RAdm. Daniel Hagari walk through one of Hamas' subterranean terrorist tunnels, only to exit in Gaza's Rantisi hospital on the other side. Inside these tunnels, Hamas terrorists hide, operate and hold Israeli hostages against their will.",
     "Watch IDF Spokesperson RAdm. Daniel Hagari walk through one of Hamas' subterranean tunnels, only to exit in Gaza's Rantisi hospital on the other side. Inside these tunnels, he says, Hamas members hide, operate and hold Israeli hostages against their will.",
     "loaded label; unattributed claim", "R02;R10"),
    (30, "against_palestine",
     "Yocheved Lifshitz, one of two hostages recently freed by Hamas, has described her ordeal after being kidnapped by the Palestinian militant group in the brazen October 7 attack, saying she \"went through hell.\"",
     "Yocheved Lifshitz, one of two hostages recently freed by Hamas, has described her ordeal after being kidnapped by the Palestinian militant group in the October 7 attack, saying she \"went through hell.\"",
     "intensifier (quote kept)", "R03"),
    (31, "against_palestine",
     "The terrorists were from East Jerusalem, one of them had been imprisoned in Israel previously.",
     "The attackers were from East Jerusalem, one of them had been imprisoned in Israel previously.",
     "loaded label; evidence-only", "R02;R01"),
    (32, "against_palestine",
     "Five brave young survivors of October 7, now scattered across Israel, tell the story of the children who survived the shocking massacre.",
     "Five young survivors of October 7, now scattered across Israel, tell the story of the children who survived the attack.",
     "intensifier; massacre", "R03;R18"),
    (33, "against_palestine",
     "BEAUTY AND THE BEASTS: Former Miss World scorches the United Nations and women's groups refusing to condemn Hamas terrorists' brutal atrocities.",
     "Former Miss World criticizes the United Nations and women's groups refusing to condemn atrocities by Hamas.",
     "dehumanizing; loaded label; intensifier", "R04;R02;R03"),
    (34, "against_palestine",
     "Second group of hostages released from grip of Hamas terrorists in Gaza.",
     "Second group of hostages released by Hamas in Gaza.",
     "loaded label", "R02"),
    (35, "against_palestine",
     "Hamas militants targeted the area in the deadly terror attack last month.",
     "Hamas militants targeted the area in the deadly attack last month.",
     "contested label", "R20;R02"),
    (36, "against_israel",
     "Dozens of Australian doctors have been reported to the country's medical regulator following complaints over their social media posts on the war on Gaza and use of the term \"genocide\".",
     "Dozens of Australian doctors have been reported to the country's medical regulator following complaints over their social media posts on the war in Gaza and use of the term \"genocide\".",
     "war framing", "R21"),
    (37, "against_israel",
     "A video circulating on social media shows Israeli soldiers joking in the destroyed remains of what appears to be a child's bedroom belonging to Palestinians who were forcibly displaced from their home due to Israel's assault on the Gaza Strip.",
     "A video circulating on social media shows Israeli soldiers joking in the destroyed remains of what appears to be a child's bedroom belonging to Palestinians who were forcibly displaced from their home due to Israel's military campaign in the Gaza Strip.",
     "war framing", "R21"),
    (38, "against_israel",
     "As Israel continues to pound Gaza and conducts raids in the occupied West Bank, Hamas says there will be no deal on captives until the 'aggression' stops.",
     "As Israel continues to strike Gaza and conducts raids in the occupied West Bank, Hamas says there will be no deal on captives until the 'aggression' stops.",
     "intensifier (quote kept)", "R03"),
    (39, "against_israel",
     "On Wednesday, compelling images surfaced from Khan Younis in southern Gaza, revealing the mass burial of Palestinians who lost their lives to the devastating airstrikes and ensuing fires.",
     "On Wednesday, images surfaced from Khan Younis in southern Gaza, showing the mass burial of Palestinians who were killed in the airstrikes and ensuing fires.",
     "editorial word; intensifier; presupposing verb; euphemism", "R17;R03;R06;R23"),
    (40, "against_israel",
     "The Israeli military is carrying out a 'precise and targeted operation against Hamas' at al-Shifa Hospital, despite no solid evidence presented of Hamas operating there.",
     "The Israeli military says it is carrying out a 'precise and targeted operation against Hamas' at al-Shifa Hospital; no solid evidence has been presented of Hamas operating there.",
     "quote without speaker; framing", "R07;R10"),
    (41, "against_israel",
     "We've seen Israel's hostages coming home in a blaze of coverage, but very little of Palestinian prisoners returning to Jerusalem as part of the truce - there's a good reason for that.",
     "We've seen Israel's hostages coming home with wide coverage, but very little of Palestinian prisoners returning to Jerusalem as part of the truce.",
     "editorial commentary; unattributed judgment", "R17;R10"),
    (42, "against_israel",
     "Israel's renewed strikes on Gaza have killed hundreds of people since Friday, Gazan officials say, raising questions about Israel's stated efforts to reduce harm to civilians.",
     "Israel's renewed strikes on Gaza have killed hundreds of people since Friday, Gazan officials say.",
     "unattributed judgment", "R10"),
    (43, "against_israel",
     "Israel pounded targets in the southern Gaza Strip on Saturday, intensifying a renewed offensive that followed a weeklong truce with Hamas and giving rise to renewed concerns about civilian casualties.",
     "Israel struck targets in the southern Gaza Strip on Saturday, intensifying a renewed offensive that followed a weeklong truce with Hamas and giving rise to renewed concerns about civilian casualties.",
     "intensifier", "R03"),
    (44, "against_israel",
     "Israel's military is ignoring international appeals to spare civilians as it hit more than 400 'Hamas targets' across Gaza over the past day, using air strikes and shelling from tanks and navy gunships.",
     "Despite international appeals to spare civilians, Israel's military hit more than 400 targets across Gaza over the past day that it says belonged to Hamas, using air strikes and shelling from tanks and navy gunships.",
     "motive framing; scare quotes", "R07;R10"),
    (45, "against_israel",
     "A new report by journalist Yuval Abraham has revealed that the Israeli army has used artificial intelligence to create civilian destruction as a military tactic to put civilian pressure on Hamas.",
     "A new report by journalist Yuval Abraham says that the Israeli army has used artificial intelligence to create civilian destruction as a military tactic to put civilian pressure on Hamas.",
     "presupposing verb", "R06"),
    (46, "against_israel",
     "Israeli officials spent months undermining the Palestinian Ministry of Health's data on killings in Gaza, and Western media has downplayed the ministry's credibility by calling it \"Hamas-run.\"",
     "Israeli officials spent months questioning the Palestinian Ministry of Health's data on killings in Gaza, and Western media has called the ministry \"Hamas-run.\"",
     "presupposing verbs; unattributed judgment", "R06;R10"),
    (47, "against_israel",
     "Hamas has invited Elon Musk to witness in person the scope of the violence and devastation heaped upon the Gaza Strip by Israel.",
     "Hamas has invited Elon Musk to witness in person the destruction caused in the Gaza Strip by Israel.",
     "intensifier", "R03"),
    (48, "against_israel",
     "Israeli airstrikes continued to pound the Gaza Strip while Hamas kept firing rockets at Israel on Saturday, the 16th day of the ongoing armed conflict.",
     "Israeli airstrikes continued to strike the Gaza Strip while Hamas kept firing rockets at Israel on Saturday, the 16th day of the ongoing armed conflict.",
     "intensifier", "R03"),
    (49, "against_israel",
     "The International Court of Justice has delivered a rebuke to the way Israel has waged war in Gaza, and to the conduct of Binyamin Netanyahu's government.",
     "The International Court of Justice has issued a ruling on the way Israel has waged war in Gaza, and on the conduct of Binyamin Netanyahu's government.",
     "court ruling as narrator verdict", "R20"),
    (50, "against_israel",
     "The conflict in Gaza has reached a grim milestone, with the Hamas-run authority reporting over 18,000 Palestinians killed, while Israeli tanks and military forces continue to fight to the center of Khan Younis.",
     "The conflict in Gaza has reached a milestone, with the Hamas-run authority reporting over 18,000 Palestinians killed, while Israeli tanks and military forces continue to fight to the center of Khan Younis.",
     "intensifier", "R03"),
]
NEW_FLAGGED = {
    20: "horrors massacre", 21: "terrorist group", 22: "massacred", 23: "HEARTBREAKING terrorists",
    24: "terror attacks", 25: "brutality", 26: "terrorists war on", 27: "terrorist organization claiming",
    28: "terrorists", 29: "terrorist tunnels terrorists", 30: "brazen", 31: "terrorists",
    32: "brave shocking massacre", 33: "BEASTS terrorists brutal scorches", 34: "grip terrorists",
    35: "terror attack", 36: "war on Gaza", 37: "assault on", 38: "pound",
    39: "compelling revealing lost their lives devastating",
    40: "precise targeted despite", 41: "blaze good reason", 42: "raising questions stated", 43: "pounded",
    44: "ignoring Hamas targets", 45: "revealed", 46: "undermining downplayed", 47: "violence devastation heaped",
    48: "pound", 49: "rebuke", 50: "grim",
}

new = pd.DataFrame(NEW_ROWS, columns=["id", "side", "biased", "neutral", "bias_type", "expected_rules"])
new.insert(1, "conflict", "israel_palestine")
tests = pd.concat([tests[tests["id"] < 20], new], ignore_index=True)   # safe to rerun
assert tests["id"].is_unique and set(NEW_FLAGGED) == set(new["id"])
print(tests["side"].value_counts())

In [ ]:
# C7d. Recall: tuning vs held-out, per side (flagged-word queries)
FLAGGED.update(NEW_FLAGGED)
assert set(FLAGGED) == set(tests["id"])

rows, heldout_misses = [], []
for set_name, part in (("tuning (1-19)", tests[tests["id"] <= 19]), ("held-out (20-50)", tests[tests["id"] >= 20])):
    for side in ("both", "against_palestine", "against_israel"):
        sub = part if side == "both" else part[part["side"] == side]
        row = {"set": set_name, "side": side, "n": len(sub)}
        for k in (3, 4):
            score, misses = recall(sub, "flagged", k)
            row[f"k={k}"] = f"{round(score * len(sub))}/{len(sub)} ({score:.0%})"
            if set_name.startswith("held") and side == "both" and k == 3:
                heldout_misses = misses
        rows.append(row)

summary = pd.DataFrame(rows)
summary

In [ ]:
# Held-out misses at k=3: which rules were expected and which were retrieved instead
pd.DataFrame(heldout_misses)

In [ ]:
out = utils.save_result(
    "guidebook_recall_heldout",
    {f"{r['set']}|{r['side']}|{k}": r[k] for r in rows for k in ("k=3", "k=4")},
    extra={"n_tuning": int((tests["id"] <= 19).sum()), "n_heldout": int((tests["id"] >= 20).sum()),
           "guidebook": str(config.GUIDEBOOK_PATH), "embedding_model": config.EMBEDDING_MODEL,
           "flagged_words": "hand-written stand-in for the detector",
           "note": "no rule changed after the held-out sentences were added"},
)
print("saved:", out)

## C8. Iteration log and findings

| Version | Change | flagged k=3 | flagged k=4 | sentence k=3 | sentence k=4 |
|---|---|---|---|---|---|
| v1 | First draft, 25 rules | not run | not run | 7/19 | 8/19 |
| v2 | R16 reworded: it named Gazans, Israelis, Russians and Ukrainians, so it matched almost every conflict sentence | 17/19 | 18/19 | 8/19 | 8/19 |
| v3 | R02 adds "terrorist", "terror group"; R14 adds "stolen land" to its Rule line | 18/19 | **19/19** | 8/19 | 8/19 |
| v4 | Rule-to-source table added above the entries; R02 allows legal designations as attributed facts | 18/19 | 19/19 | 8/19 | 9/19 |
| v5 | 31 held-out sentences added (ids 20 to 50); **no rule changes** | tuning 18/19, held-out 18/31 | tuning 19/19, held-out 19/31 | not run | not run |

**Findings from the held-out set (v5)**

- **Recall drops on new text:** 58% at k=3 on the held-out set vs 95% on the tuning set. This is the expected
  cost of tuning on a small set, and the reason the held-out set exists.
- **Recall is uneven between the sides:** at k=3, 12/16 (75%) for bias against Palestine vs 6/15 (40%) for bias
  against Israel. The system would therefore find the right rule more reliably for one side than the other.
- **Why:** bias against Palestine in FigNews is mostly loaded labels ("terrorists", "massacre", "brutal"), which the
  rules name directly. Bias against Israel is mostly judgments and framing verbs ("is ignoring", "undermining",
  "raising questions", "rebuke", "pound"), which the rules mostly do not list. R10 (unattributed opinions) is
  expected in 6 held-out sentences and never retrieved.
- **Data note:** in FigNews, bias against Israel is mostly in *what* is reported rather than *how*: only 37 of 141
  new agreed "against Israel" posts had word-level cues, so balancing the sides took a manual search of uncued posts.

**Next step (decision for the team):** rewording R03, R10, R20 and R21 would likely raise these numbers, but then
ids 20 to 50 would no longer be held-out. Any such change is logged as v6, and v6 is evaluated on a set it has
not seen (the Russia-Ukraine sentences).

## C9. Symmetry review sheet
Every rule must fix bias against **either** side. This prints each rule with its example so a teammate can check:
*would this rule apply the same way if Side A and Side B were swapped?*
It also lists which rules each side's test sentences use, to spot rules tested on one side only.

In [ ]:
for e in entries:
    print(e, "\n")

In [ ]:
used = (tests.assign(rule=tests["expected_rules"].str.split(";")).explode("rule")
             .groupby(["rule", "side"]).size().unstack(fill_value=0))
used

Rules tested on one side only are not a problem in themselves (real data is uneven), but the Russia-Ukraine
sentences and the fictional swap pairs should cover the other side where possible.

## C10. Save the final files
The team format for `data/test_sentences.csv` is `id, conflict, biased, neutral, expected_rules`; the helper
columns `side` and `bias_type` are kept only in the full copy on Drive. Ids 1 to 19 are the tuning set and ids
20 to 50 the held-out set. Download `test_sentences.csv` from `drive/MyDrive/debias` (not from the repo's `data/`
folder) and upload it to a new branch, e.g. `amen/test-sentences-50`.

In [ ]:
tests.to_csv(ARTIFACTS / "test_sentences_full.csv", index=False)                    # with side and bias_type
final = tests[["id", "conflict", "biased", "neutral", "expected_rules"]]
final.to_csv(ARTIFACTS / "test_sentences.csv", index=False)                         # team format

print(final.shape, "->", ARTIFACTS / "test_sentences.csv")
print("guidebook   ->", config.GUIDEBOOK_PATH)
assert len(final) == 50 and final["id"].tolist() == list(range(1, 51)), "expected ids 1 to 50"